# CityPulse Accelerated — GPU Benchmark (Colab)

Compare **pandas** vs **NVIDIA RAPIDS cuDF** pipeline timing at 50K, 500K, and 2M synthetic complaint rows.

## Before you start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all** (run cells **top to bottom** — do not skip Step 2)

Repo: https://github.com/ayyappancsking-byte/citypulse-accelerated

In [ ]:
!rm -rf citypulse-accelerated

In [ ]:
# Step 1 — Confirm GPU is active
!nvidia-smi

In [ ]:
# Step 2 — Clone repo and enter project root (required before any other step)
import os
from pathlib import Path

REPO_URL = "https://github.com/ayyappancsking-byte/citypulse-accelerated.git"
PROJECT_DIR = "citypulse-accelerated"

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL}
elif not (Path(PROJECT_DIR) / "pipeline").exists():
    !rm -rf {PROJECT_DIR}
    !git clone {REPO_URL}

%cd {PROJECT_DIR}

if not Path("pipeline").exists():
    raise RuntimeError(
        "Could not find pipeline/ folder. "
        "Make sure the repo is public and run this cell again."
    )

print("Project root:", Path.cwd())
print("Files:", sorted(os.listdir("."))[:12])

In [ ]:
# Step 3 — Install RAPIDS cuDF + project dependencies (~2–4 min)
!pip install -q cudf-cu12 --extra-index-url=https://pypi.nvidia.com
!pip install -q faker pyarrow matplotlib python-dotenv google-genai

In [ ]:
# Step 4 — Generate synthetic data (50K / 500K / 2M rows, ~3–8 min)
!python data/generate_synthetic_data.py --sizes 50000 500000 2000000

In [ ]:
# Step 5 — Sanity check: run scoring pipeline on 50K dataset
!python -m pipeline.run_pipeline --data data/complaints_50k.csv

In [ ]:
# Step 6 — Run pandas vs cuDF benchmark (~5–15 min)
!python -m pipeline.benchmark

In [ ]:
# Step 7 — View results (Slide 9 chart + demo sentence)
from IPython.display import Image, display
from pathlib import Path

chart = Path("outputs/benchmark_chart.png")
takeaway = Path("outputs/benchmark_takeaway.txt")

if takeaway.exists():
    print(takeaway.read_text(encoding="utf-8"))

if chart.exists():
    display(Image(filename=str(chart)))
else:
    print("Chart not generated — check benchmark_results.csv for errors.")

In [ ]:
# Step 8 — Download outputs to your PC
from google.colab import files

for path in [
    "outputs/benchmark_chart.png",
    "outputs/benchmark_results.csv",
    "outputs/benchmark_takeaway.txt",
]:
    print(f"Downloading {path}...")
    files.download(path)

print("\nDone! Use benchmark_chart.png on Slide 9 of your deck.")